# 1. Tujuan dan sumber data

Notebook lokal ini mengikuti urutan **EDA → prapemrosesan → rekayasa fitur**. Sumbernya hanya konfigurasi `main` IndoDiscourse pada revisi tetap. Konfigurasi `annotator` terpisah tidak dimuat. Jumlah saat ini adalah 28.448 baris main dan 29 baris annotator; agregat tingkat dataset 28.477 mencakup keduanya. Pengamatan ini belum memastikan makna angka dalam artikel atau kartu dataset. Data berasal dari sumber yang ditargetkan (X, Facebook, Instagram, dan berita), sehingga ringkasan ini tidak menunjukkan keterwakilan populasi.

Cache dan hasil disimpan secara lokal di `data/`, yang diabaikan Git. Jangan membagikan data atau membuat mirror publik sebelum hak redistribusi jelas. Tidak ada pemuatan tokenizer/model, pembagian fold, pelatihan, atau evaluasi di sini. Jalankan sel secara berurutan dari awal.


In [ ]:
import hashlib, json, os, platform, re, tempfile, tomllib, unicodedata, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

def has_project_markers(path):
    return (path / "pyproject.toml").is_file() and (path / "kaggle" / "config" / "experiment.toml").is_file()


start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents) if has_project_markers(candidate)), None)
if root is None and has_project_markers(start / "project"):
    root = start / "project"
if root is None:
    raise RuntimeError("Jalankan Jupyter dari folder repo, folder notebooks, atau workspace yang berisi folder project.")

config_path = root / "kaggle" / "config" / "experiment.toml"
with config_path.open("rb") as f:
    experiment = tomllib.load(f)
dataset_config = experiment["dataset"]
source_schema = dataset_config["schema"]
expected_columns = list(source_schema)
revision = dataset_config["revision"]
filename = dataset_config["filename"]
expected_rows = dataset_config["main_rows"]
expected_sha256 = dataset_config["sha256"]
url = f"https://huggingface.co/datasets/{dataset_config['repo_id']}/resolve/{revision}/{filename}"
raw_dir, processed_dir = root / "data" / "raw", root / "data" / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
cache = raw_dir / f"{revision}_{filename}"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if not cache.exists():
    with tempfile.TemporaryDirectory(dir=raw_dir, prefix=".download-") as temp_dir:
        tmp_path = Path(temp_dir) / filename
        try:
            with tmp_path.open("wb") as tmp:
                with urllib.request.urlopen(url, timeout=120) as response:
                    while chunk := response.read(1024 * 1024):
                        tmp.write(chunk)
        except Exception as exc:
            raise RuntimeError("Unduhan gagal. Periksa koneksi dan akses ke URL dataset, lalu jalankan kembali sel ini.") from exc
        if sha256_file(tmp_path) != expected_sha256:
            raise ValueError("SHA-256 unduhan tidak sesuai revisi HF; file tidak dipindahkan ke cache.")
        os.replace(tmp_path, cache)
raw_sha256 = sha256_file(cache)
if raw_sha256 != expected_sha256:
    raise ValueError("SHA-256 cache tidak sesuai revisi HF. Periksa atau hapus cache yang rusak di data/raw lalu jalankan ulang.")
print("Cache tervalidasi; SHA-256:", raw_sha256)


# 2. EDA: skema dan validasi

Setiap baris adalah satu pengamatan. Validasi menghentikan proses jika skema, jumlah baris, tipe teks, daftar ID, suara biner, atau kesesuaian panjang tidak memenuhi aturan. Laporan kesalahan hanya memuat indeks baris dan nama kolom; nilai data tidak ditampilkan. ID anotator dipakai sementara untuk memeriksa panjang dan menghitung jumlah anotasi.

Pada revisi yang dipin, baris indeks 10.295 memiliki `text` null, meskipun pasangan suaranya `(0, 0)`. Proposal mensyaratkan teks berupa string dan menghentikan proses pada data yang bentuknya tidak sesuai. Karena itu notebook sengaja berhenti di validasi; nilai null tidak diubah menjadi string kosong atau dihapus otomatis. Lanjutkan preprocessing setelah protokol metode menetapkan penanganan baris ini.


In [ ]:
def matches_schema(value, expected_type):
    if expected_type == "string":
        return isinstance(value, str)
    if expected_type == "list[string]":
        return isinstance(value, list) and all(isinstance(item, str) for item in value)
    return False


try:
    with cache.open(encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
except (OSError, UnicodeError, json.JSONDecodeError) as exc:
    raise RuntimeError("Cache tidak dapat dibaca sebagai JSONL; periksa atau hapus cache lokal lalu unduh ulang.") from exc
if len(records) != expected_rows:
    raise ValueError(f"Jumlah baris cache {len(records)}, diharapkan {expected_rows}; periksa cache lokal.")
errors = []
for i, row in enumerate(records):
    if not isinstance(row, dict):
        errors.append((i, "<row>"))
        continue
    if set(row) != set(expected_columns):
        errors.append((i, "<columns>"))
    for field, expected_type in source_schema.items():
        value = row.get(field)
        if value is None and field not in {"text_id", "annotators_id", "text", "toxicity", "polarized"}:
            continue
        if field in row and not matches_schema(value, expected_type):
            errors.append((i, field))
    text_id = row.get("text_id")
    if isinstance(text_id, str) and not text_id.strip():
        errors.append((i, "text_id:empty"))
    ids = row.get("annotators_id")
    if isinstance(ids, list) and (not ids or any(not isinstance(item, str) or not item.strip() for item in ids)):
        errors.append((i, "annotators_id:empty"))
    for field in ("toxicity", "polarized"):
        votes = row.get(field)
        if isinstance(votes, list) and (not votes or any(vote not in ("0", "1") for vote in votes)):
            errors.append((i, field + ":binary"))
        elif isinstance(votes, list) and isinstance(ids, list) and len(votes) != len(ids):
            errors.append((i, field + ":length"))
if errors:
    summary = pd.DataFrame(errors, columns=["indeks_baris", "kolom"]).drop_duplicates()
    raise ValueError("Validasi gagal (indeks 0-based dan kolom):\n" + summary.to_string(index=False))
df = pd.DataFrame.from_records(records, columns=expected_columns)
del records
def is_missing_value(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value == ""
    if isinstance(value, list):
        return not value
    return bool(pd.isna(value))


missing_values = {column: int(sum(is_missing_value(value) for value in df[column])) for column in expected_columns}
schema = pd.DataFrame({"kolom": expected_columns, "tipe_sumber": [source_schema[c] for c in expected_columns], "nilai_hilang_atau_kosong": [missing_values[c] for c in expected_columns]})
df["annotation_count"] = df["annotators_id"].map(len)
df = df.drop(columns="annotators_id")
display(schema)
print("Baris:", len(df), "Kolom sumber:", len(expected_columns))


# 3. EDA: panjang teks dan anotasi

Statistik dan grafik berikut bersifat agregat. Panjang karakter menghitung titik kode Python; jumlah kata berbasis pemisahan whitespace bukan jumlah subword IndoBERTweet. Tidak ada teks atau ID yang ditampilkan.


In [ ]:
lengths = pd.DataFrame({"karakter": df.text.str.len(), "token_whitespace": df.text.map(lambda x: len(x.split())), "jumlah_anotasi": df.annotation_count})
display(lengths.describe().T)
lengths.hist(bins=30, figsize=(12, 3)); plt.tight_layout(); plt.show()
print("Teks hanya whitespace (tidak termasuk string kosong):", int(df.text.map(lambda x: bool(x) and not x.strip()).sum()))


# 4. Prapemrosesan: suara mayoritas dan pemilihan sampel

Mayoritas ketat dihitung terpisah untuk kedua tugas. Seri pada salah satu tugas dikeluarkan terlebih dahulu. Setelah itu, hanya string kosong persis `""` dengan pasangan `(0, 0)` yang dikeluarkan; teks berisi whitespace tetap dipertahankan. Noise/spam dan duplikat tetap ada. Ledger memakai kategori yang tidak tumpang tindih.


In [ ]:
def majority(votes):
    ones = votes.count("1")
    return None if 2 * ones == len(votes) else int(2 * ones > len(votes))

work = df.copy()
work["toxicity_label"] = work.toxicity.map(majority)
work["polarized_label"] = work.polarized.map(majority)
tie = work[["toxicity_label", "polarized_label"]].isna().any(axis=1)
empty_exclusion = (~tie) & work.text.eq("") & work.toxicity_label.eq(0) & work.polarized_label.eq(0)
keep = ~(tie | empty_exclusion)
ledger = pd.Series({"awal": len(work), "dikeluarkan_seri": int(tie.sum()), "dikeluarkan_kosong_00": int(empty_exclusion.sum()), "dipertahankan": int(keep.sum())}, name="baris")
assert ledger["awal"] == ledger["dikeluarkan_seri"] + ledger["dikeluarkan_kosong_00"] + ledger["dipertahankan"]
work = work.loc[keep].copy()
work[["toxicity_label", "polarized_label"]] = work[["toxicity_label", "polarized_label"]].astype(int)
work["row_id"] = work.index
display(ledger.to_frame())


# 5. Audit label terhadap proposal

Tabel marginal dan pasangan menggambarkan sampel aktual. Referensi proposal adalah 24.855 baris; selisih ditampilkan sebagai aktual dikurangi referensi, tanpa mengubah atau menyeimbangkan data.


In [ ]:
for col in ["toxicity_label", "polarized_label"]:
    print(col)
    display(work[col].value_counts().reindex([0, 1], fill_value=0).rename("baris").to_frame())
pairs = work.groupby(["toxicity_label", "polarized_label"]).size().reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=["toxicity_label", "polarized_label"]), fill_value=0)
comparison = pd.DataFrame({"aktual": pairs, "proposal": [20436, 2591, 933, 895]})
comparison["selisih"] = comparison.aktual - comparison.proposal
display(comparison)
print("Total aktual, proposal, selisih:", len(work), 24855, len(work)-24855)
pairs.plot.bar(rot=0, title="Pasangan label aktual"); plt.ylabel("Baris"); plt.tight_layout(); plt.show()


# 6. Rekayasa fitur dan kelompok duplikat

`group_key` dibentuk dengan NFKC, casefold, lalu menghapus dan merangkum whitespace. Baris duplikat tidak dihapus. Kunci hanya metadata pembagian di tahap berikutnya: rencananya lima fold StratifiedGroupKFold berdasarkan pasangan label, seed 42, tetapi fold belum dibuat. Fitur hitungan ini hanya diagnostik, bukan masukan classifier. Tidak ada pembersihan manual tanda baca, URL, kata kasar, emoji, atau bahasa campuran.


In [ ]:
work["group_key"] = work.text.map(lambda s: " ".join(unicodedata.normalize("NFKC", s).casefold().split()))
work["char_count"] = work.text.str.len()
work["whitespace_token_count"] = work.text.map(lambda s: len(s.split()))
work["punctuation_count"] = work.text.map(lambda s: sum(unicodedata.category(c).startswith("P") for c in s))
work["digit_count"] = work.text.map(lambda s: sum(c.isdigit() for c in s))
work["url_count"] = work.text.map(lambda s: len(re.findall(r"https?://[^\s]+|www\.[^\s]+", s, flags=re.I)))
work["mention_count"] = work.text.map(lambda s: len(re.findall(r"(?<!\w)@[\w]+", s)))
work["hashtag_count"] = work.text.map(lambda s: len(re.findall(r"(?<!\w)#[\w]+", s)))
features = ["char_count", "whitespace_token_count", "punctuation_count", "digit_count", "url_count", "mention_count", "hashtag_count"]
groups = work.groupby("group_key", sort=False, dropna=False)
sizes = groups.size()
conflicts = groups[["toxicity_label", "polarized_label"]].nunique().gt(1).any(axis=1)
print("Jumlah kelompok:", len(sizes), "kelompok >1 baris:", int(sizes.gt(1).sum()), "kelompok dengan label konflik:", int(conflicts.sum()))
display(sizes.value_counts().sort_index().rename_axis("ukuran_kelompok").rename("jumlah_kelompok").to_frame())
display(work[features].describe().T)
work[features].hist(bins=30, figsize=(12, 8)); plt.tight_layout(); plt.show()


# 7. Simpan hasil lokal dan manifest

JSONL turunan menyimpan teks asli, ID baris lokal, dua label, kunci kelompok, dan fitur diagnostik. ID anotator dan demografi tidak disimpan. Pada tahap model nantinya, hanya teks asli yang menjadi masukan classifier; ID baris, kunci, fitur diagnostik, dan label bukan fitur masukan. Manifest mencatat asal, hash cache, jumlah, dan versi lingkungan. Hasil lokal ini mengandung teks sumber dan tidak boleh dipublikasikan tanpa kepastian hak redistribusi.


In [ ]:
output = processed_dir / "indodiscourse_main_preprocessed.jsonl"
manifest_path = processed_dir / "indodiscourse_main_manifest.json"
columns = ["row_id", "text", "toxicity_label", "polarized_label", "group_key"] + features
work[columns].to_json(output, orient="records", lines=True, force_ascii=False)
manifest = {
    "dataset": dataset_config["repo_id"],
    "configuration": dataset_config["configuration"],
    "revision": revision,
    "url": url,
    "filename": filename,
    "expected_sha256": expected_sha256,
    "raw_sha256": raw_sha256,
    "raw_rows": len(df),
    "processed_rows": len(work),
    "source_schema": source_schema,
    "source_missing_values": missing_values,
    "derived_schema": {
        "row_id": "integer",
        "text": "string",
        "toxicity_label": "integer",
        "polarized_label": "integer",
        "group_key": "string",
        **{feature: "integer" for feature in features},
    },
    "ledger": {key: int(value) for key, value in ledger.items()},
    "joint_counts": {f"{a},{b}": int(count) for (a, b), count in pairs.items()},
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Hasil dan manifest tersimpan di direktori lokal data/processed.")
